# Principal Component Analysis

In [ ]:
%run common.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

feature_names = list(train_features.columns)
n_features    = len(feature_names)

pca = PCA(n_components=n_features)
pca.fit(train_scaled)

explained_variance_ratio = pca.explained_variance_ratio_           # shape: (n_features,)
components               = pca.components_                         # shape: (n_components, n_features)
                                                                   # each row is one PC

# ---------------------------------------------------------------------------
# Figure 1 – Explained variance
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pc_labels = [f"PC{i+1}" for i in range(n_features)]

# ── left panel: per-component explained variance ──────────────────────────
axes[0].bar(
    pc_labels,
    explained_variance_ratio * 100,
    color="steelblue",
    edgecolor="white",
)
axes[0].set_xlabel("Principal Component", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Explained Variance (%)", fontsize=12, fontweight="bold")
axes[0].set_title("Explained Variance per Principal Component",
                  fontsize=14, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# ── right panel: cumulative explained variance ────────────────────────────
cumulative = np.cumsum(explained_variance_ratio) * 100
axes[1].plot(pc_labels, cumulative, marker="o", color="darkorange", linewidth=2)
axes[1].axhline(y=95, color="red", linestyle="--", linewidth=1, label="95 % threshold")
axes[1].set_xlabel("Principal Component", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Cumulative Explained Variance (%)", fontsize=12, fontweight="bold")
axes[1].set_title("Cumulative Explained Variance",
                  fontsize=14, fontweight="bold")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0, 105])

plt.tight_layout()
plt.show()

print("=" * 70)
print("EXPLAINED VARIANCE PER PRINCIPAL COMPONENT")
print("=" * 70)
for i, (pc, ev, cum) in enumerate(
    zip(pc_labels, explained_variance_ratio * 100, cumulative)
):
    print(f"  {pc}: {ev:6.2f}%  (cumulative: {cum:6.2f}%)")

# ---------------------------------------------------------------------------
# Figure 2 – Feature loadings heat-map
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(max(8, n_features * 1.6), max(4, n_features * 0.9)))

im = ax.imshow(components, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Loading")

ax.set_xticks(range(n_features))
ax.set_xticklabels(feature_names, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(n_features))
ax.set_yticklabels(pc_labels, fontsize=10)

# Annotate each cell with the loading value
for row in range(n_features):
    for col in range(n_features):
        ax.text(
            col, row,
            f"{components[row, col]:.2f}",
            ha="center", va="center",
            fontsize=8,
            color="black",
        )

ax.set_title("PCA Feature Loadings  (component × feature)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Original Feature", fontsize=12, fontweight="bold")
ax.set_ylabel("Principal Component", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# Figure 3 – PCA-based feature importance
#
# For every original feature f, the importance is:
#
#     importance(f) = Σ_k  explained_variance_ratio[k] × loadings[k, f]²
#
# This is the variance of the *original* feature explained by the PCA model,
# weighted by how much each PC itself explains of the total variance.
# Values sum to 1 across all features.
# ---------------------------------------------------------------------------

importance = np.zeros(n_features)
for k in range(n_features):
    importance += explained_variance_ratio[k] * (components[k] ** 2)

# Normalise so scores sum to 1 (they already should, but normalise for safety)
importance = importance / importance.sum()

# Sort descending for the bar chart
order         = np.argsort(importance)[::-1]
sorted_names  = [feature_names[i] for i in order]
sorted_scores = importance[order]

fig, ax = plt.subplots(figsize=(max(8, n_features * 1.4), 5))

bars = ax.bar(
    sorted_names,
    sorted_scores * 100,
    color="steelblue",
    edgecolor="white",
)
ax.set_xlabel("Feature", fontsize=12, fontweight="bold")
ax.set_ylabel("PCA-based Importance (%)", fontsize=12, fontweight="bold")
ax.set_title("Feature Importance Derived from PCA\n"
             r"(Σ explained_var_ratio × loading²,  normalised)",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")

# Label each bar
for bar, score in zip(bars, sorted_scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{score * 100:.1f}%",
        ha="center", va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# Textual summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("PCA-BASED FEATURE IMPORTANCE (sorted)")
print("=" * 70)
for name, score in zip(sorted_names, sorted_scores):
    bar = "█" * int(round(score * 50))
    print(f"  {name:<35s}  {score * 100:5.1f}%  {bar}")
